In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

/export/usuarios01/zhuldyz/Zhuldyz_files/DiffuSeq/.diffseq/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [38]:
import json
import shutil

full_path = "datasets/QQP/test_full.jsonl"
data_path = "datasets/QQP/test.jsonl"

# Read all samples from the original test file
with open(data_path, "r") as f:
    samples = [json.loads(line) for line in f]

# Save the full version as test_full
shutil.copy(data_path, full_path)

# Overwrite test with only 20 samples
with open(data_path, "w") as f:
    for sample in samples[:20]:
        f.write(json.dumps(sample) + "\n")

print(f"Saved {len(samples)} samples to {full_path}")
print(f"Saved 20 samples to {data_path}")

Saved 2500 samples to datasets/QQP/test_full.jsonl
Saved 20 samples to datasets/QQP/test.jsonl


In [39]:
import json
from pathlib import Path

def load_json_or_jsonl(path):
    path = Path(path)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    if not text:
        return []

    # First try normal JSON
    try:
        data = json.loads(text)
        if isinstance(data, dict) and "data" in data:
            return data["data"]
        if isinstance(data, list):
            return data
        return [data]
    except json.JSONDecodeError:
        pass

    # Fallback: JSON Lines
    records = []
    for line in text.splitlines():
        line = line.strip()
        if line:
            records.append(json.loads(line))
    return records


In [40]:
def flatten_recovery_records(records):
    rows = []

    for i, rec in enumerate(records):
        sequence_id = rec.get("sequence_id", f"seq_{i:06d}")
        source = rec.get("source", "")
        reference = rec.get("reference", "")

        for step in rec.get("recover_steps", []):
            rows.append({
                "sequence_id": sequence_id,
                "source": source,
                "reference": reference,
                "step_index": step.get("step_index"),
                "timestep": step.get("timestep"),
                "recover": step.get("recover", "")
            })

    return pd.DataFrame(rows)

In [53]:
records = load_json_or_jsonl("generation_outputs/diffuseq_qqp_h128_lr0.0001_t2000_sqrt_lossaware_seed102_qqp20260503-19:48:44/ema_0.9999_005000.pt.samples/seed123_step0.intermediate.json")
df = flatten_recovery_records(records)
print(list(records[0].keys()))

['source', 'reference', 'recover_steps']


In [34]:
SPECIAL_TOKENS = [
    "[CLS]",
    "[SEP]",
    "[PAD]",
    "[UNK]",
    "[MASK]"
]

def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    for tok in SPECIAL_TOKENS:
        text = text.replace(tok, " ")

    text = re.sub(r"\s+", " ", text).strip()
    return text

In [35]:
df["source_clean"] = df["source"].apply(clean_text)
df["reference_clean"] = df["reference"].apply(clean_text)
df["recover_clean"] = df["recover"].apply(clean_text)

df[["sequence_id", "step_index", "timestep", "recover_clean"]].head()

,sequence_id,step_index,timestep,recover_clean
0,seq_000000,0,1999,
1,seq_000000,1,1998,
2,seq_000000,2,1997,
3,seq_000000,3,1996,
4,seq_000000,4,1995,


In [16]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed_unique_texts(df, text_columns, batch_size=128):
    unique_texts = pd.unique(
        pd.concat([df[col] for col in text_columns], ignore_index=True)
        .dropna()
        .astype(str)
    )

    unique_texts = [t for t in unique_texts if t.strip()]

    embeddings = model.encode(
        unique_texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    return dict(zip(unique_texts, embeddings))

In [36]:
embedding_lookup = embed_unique_texts(
    df,
    text_columns=["source_clean", "reference_clean", "recover_clean"],
    batch_size=128
)

def cosine_from_lookup(df, left_col, right_col, embedding_lookup):
    scores = np.full(len(df), np.nan)

    mask = (
        df[left_col].astype(str).str.strip().ne("") &
        df[right_col].astype(str).str.strip().ne("")
    )

    left_embeddings = np.vstack(
        df.loc[mask, left_col].map(embedding_lookup).to_numpy()
    )

    right_embeddings = np.vstack(
        df.loc[mask, right_col].map(embedding_lookup).to_numpy()
    )

    scores[mask.to_numpy()] = np.sum(left_embeddings * right_embeddings, axis=1)

    return scores

df["sim_recover_reference"] = cosine_from_lookup(
    df,
    "recover_clean",
    "reference_clean",
    embedding_lookup
)

df["sim_recover_source"] = cosine_from_lookup(
    df,
    "recover_clean",
    "source_clean",
    embedding_lookup
)

df["sim_source_reference"] = cosine_from_lookup(
    df,
    "source_clean",
    "reference_clean",
    embedding_lookup
)

df["sim_recover_reference_pct"] = (df["sim_recover_reference"] * 100).round(2)
df["sim_recover_source_pct"] = (df["sim_recover_source"] * 100).round(2)
df["sim_source_reference_pct"] = (df["sim_source_reference"] * 100).round(2)

df["avg_reference_source_similarity_pct"] = (
    df[["sim_recover_reference_pct", "sim_recover_source_pct"]]
    .mean(axis=1)
    .round(2)
)

df["gain_over_source_reference"] = (
    df["sim_recover_reference"] - df["sim_source_reference"]
).round(4)

df["gain_over_source_reference_pct"] = (
    df["gain_over_source_reference"] * 100
).round(2)

best_by_sequence = (
    df.sort_values(
        ["sequence_id", "sim_recover_reference"],
        ascending=[True, False]
    )
    .groupby("sequence_id", as_index=False)
    .first()
)

best_by_sequence[
    [
        "sequence_id",
        "step_index",
        "timestep",
        "sim_recover_reference_pct",
        "sim_recover_source_pct",
        "recover_clean"
    ]
].head()

df.to_parquet("scored_recovery_steps_baseline_005000.parquet", index=False)
best_by_sequence.to_parquet("best_recovery_by_sequence_baseline_005000.parquet", index=False)

Batches: 100%|██████████| 1/1 [00:00<00:00, 51.08it/s]

ValueError: need at least one array to concatenate

In [27]:

path = "/export/usuarios01/zhuldyz/Zhuldyz_files/DiffuSeq/best_recovery_by_sequence_eccdouble_005000.parquet"

df = pd.read_parquet(path)
df.head()

,sequence_id,source,reference,step_index,timestep,recover,source_clean,reference_clean,recover_clean,sim_recover_reference,sim_recover_source,sim_source_reference,sim_recover_reference_pct,sim_recover_source_pct,sim_source_reference_pct,avg_reference_source_similarity_pct,gain_over_source_reference,gain_over_source_reference_pct
0,seq_000000,[CLS] astrology : i am a capricorn sun cap moo...,"[CLS] i ' m a triple capricorn ( sun, moon and...",494,1505,##ping cape gonzales cape gonzales cape gonzal...,astrology : i am a capricorn sun cap moon and ...,"i ' m a triple capricorn ( sun, moon and ascen...",##ping cape gonzales cape gonzales cape gonzal...,0.213114,0.148682,0.837468,21.31,14.87,83.75,18.09,-0.6244,-62.44
1,seq_000001,[CLS] how can i be a good geologist? [SEP] [SEP],[CLS] what should i do to be a great geologist...,1,1998,cape cape projections knowles gmina cape gonza...,how can i be a good geologist?,what should i do to be a great geologist?,cape cape projections knowles gmina cape gonza...,0.117143,0.110096,0.936959,11.71,11.01,93.70,11.36,-0.8198,-81.98
2,seq_000002,[CLS] how do i read and find my youtube commen...,[CLS] how can i see all my youtube comments? [...,149,1850,bother projections balance viper knowles balan...,how do i read and find my youtube comments?,how can i see all my youtube comments?,bother projections balance viper knowles balan...,-0.001727,0.007687,0.879711,-0.17,0.77,87.97,0.30,-0.8814,-88.14
3,seq_000003,[CLS] what can make physics easy to learn? [SE...,[CLS] how can you make physics easy to learn? ...,87,1912,disruption balance bother projections balance ...,what can make physics easy to learn?,how can you make physics easy to learn?,disruption balance bother projections balance ...,-0.002782,-0.013290,0.974253,-0.28,-1.33,97.43,-0.80,-0.9770,-97.70
4,seq_000004,[CLS] what was your first sexual experience li...,[CLS] what was your first sexual experience? [...,57,1942,storm cape cape extra balance cape knowles vip...,what was your first sexual experience like?,what was your first sexual experience?,storm cape cape extra balance cape knowles vip...,0.002967,-0.002361,0.971795,0.30,-0.24,97.18,0.03,-0.9688,-96.88
